# Out-of-fold scoring of the full headline stream

`oof_finbert_scoring.ipynb` trains the five expanding-window models and scores the **20 251
labelled** headlines in `output.csv`, which stop at 2025-03-28. The downstream pipeline needs more
than that:

| consumer | file it reads | coverage needed |
|---|---|---|
| iTransformer / LSTM training | `predicted_sentiments_finbert.csv` | 2019-01 → 2025-03 |
| backtesting (`data_loader.py:27`) | `predicted_sentiments2.csv` | 2019-01 → **2025-11** |

The sideways backtest regime runs Jan–Jun 2025, so a column ending in March 2025 cannot drive it.

This notebook scores the **entire** headline stream — all 23 365 rows of
`telegram_filtering/headlines.csv`, which is row-for-row the source of `predicted_sentiments2.csv` —
using the checkpoints already trained. No training happens here.

**Why the extension is still out-of-fold.** Labels are needed to *train*, not to *score*. Fold 5's
model is frozen on data ending 2023-04-25, so applying it to headlines from 2024-01 through
2025-11 is strictly forward prediction. The 3 067 headlines after 2025-03-28 simply never had
labels computed; that has no bearing on whether the model saw them.

Fold assignment uses the same calendar edges, with slice 5 extended to the end of the stream:

| headline date | scored by | trained on |
|---|---|---|
| 2019-01 → 2020-01 | *nothing* → `NaN` | — |
| 2020-01 → 2021-01 | fold 1 | ≤ 2019-12 |
| 2021-01 → 2022-01 | fold 2 | ≤ 2020-12 |
| 2022-01 → 2023-01 | fold 3 | ≤ 2021-09 |
| 2023-01 → 2024-01 | fold 4 | ≤ 2022-09 |
| **2024-01 → 2025-11** | fold 5 | ≤ 2023-04 |

Calibration reuses each fold's stored validation mean and standard deviation, so the stream scores
sit on exactly the same scale as the labelled-subset column.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, warnings
from transformers import BertForSequenceClassification, BertTokenizerFast

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

REPO    = Path.cwd().parent          # repository root
CLEANED = REPO / "backtesting" / "data" / "cleaned"
STREAM  = REPO / "telegram_filtering" / "headlines.csv"
OUT     = Path.cwd() / "out"
CKPT_DIR = Path.cwd() / "oof_checkpoints"

# must match the run being extended (see oof_finbert_scoring.ipynb)
RUN_TAG     = "tone3_cond"
HEAD_MODE   = "tone3"
OUTPUT_ACT  = "linear"
MODEL_NAME  = "yiyanghkust/finbert-tone"
MAX_LEN, INFER_BATCH, SEED = 128, 256, 42
USE_AMP = True

EDGES = [pd.Timestamp(e) for e in
         ["2019-01-01", "2020-01-01", "2021-01-01", "2022-01-01",
          "2023-01-01", "2024-01-01", "2024-06-01"]]

# drop-in replacements
FULL_CSV = CLEANED / f"predicted_sentiments_oof_{RUN_TAG}_full.csv"      # for *_finbert.csv
PS2_CSV  = CLEANED / f"predicted_sentiments_oof_{RUN_TAG}_ps2.csv"       # for predicted_sentiments2.csv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| run tag:", RUN_TAG)

In [ ]:
def _squash(x):
    return torch.tanh(x) if OUTPUT_ACT == "tanh" else x


class FinBERTRegressionTone3(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.regressor = nn.Linear(3, 1)

    def forward(self, input_ids, attention_mask):
        return _squash(self.regressor(
            self.bert(input_ids=input_ids, attention_mask=attention_mask).logits))


def build_model():
    torch.manual_seed(SEED)
    return FinBERTRegressionTone3().to(device)


@torch.no_grad()
def predict(model, ids, mask, idx):
    model.eval()
    out = torch.empty(len(idx), device=device)
    ix = torch.as_tensor(idx, device=device)
    for s in range(0, len(idx), INFER_BATCH):
        b = ix[s:s + INFER_BATCH]
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out[s:s + INFER_BATCH] = model(ids[b], mask[b]).squeeze(-1).float()
    return out.cpu().numpy().astype(np.float64)

In [ ]:
hl = (pd.read_csv(STREAM)
        .rename(columns={"date_unixtime": "timestamp", "text": "title", "from": "source"})
        [["timestamp", "title", "source"]])
hl["dt"] = pd.to_datetime(hl["timestamp"], unit="s")
# NOT sorted: headlines.csv row order is what predicted_sentiments2.csv aligns to, and the drop-in
# replacement must preserve it. Fold assignment is per-row, so order is irrelevant to correctness.

# fold index: 0 = before the first trainable window, 6 = 2024-06 onward (extends to the stream end)
hl["fold"] = np.searchsorted(np.array(EDGES[1:], dtype="datetime64[ns]"),
                             hl["dt"].values.astype("datetime64[ns]"), side="right")

print(f"{len(hl)} headlines, {hl['dt'].min()} -> {hl['dt'].max()}")
print()
print(hl.groupby("fold").agg(n=("dt", "size"), start=("dt", "min"), end=("dt", "max")).to_string())

In [ ]:
tok = BertTokenizerFast.from_pretrained(MODEL_NAME)
enc = tok(hl["title"].astype(str).tolist(), truncation=True,
          padding="max_length", max_length=MAX_LEN, return_tensors="pt")
IDS, MASK = enc["input_ids"].to(device), enc["attention_mask"].to(device)
print(f"tokenised -> {(IDS.nbytes + MASK.nbytes)/1e6:.0f} MB on device")

calib = pd.read_csv(OUT / f"oof_fold_calibration_{RUN_TAG}.csv").set_index("fold")
print()
print(calib[["n_val", "val_mean", "val_sd"]].to_string())

In [ ]:
raw = np.full(len(hl), np.nan)
z   = np.full(len(hl), np.nan)

for k in range(1, 7):
    tag = f"fold{k}"
    idx = np.flatnonzero(hl["fold"].to_numpy() == k)
    if len(idx) == 0:
        continue
    model = build_model()
    model.load_state_dict(torch.load(CKPT_DIR / f"finbert_oof_{RUN_TAG}_{tag}.pth",
                                     map_location=device))
    p = predict(model, IDS, MASK, idx)
    mu, sd = calib.loc[tag, "val_mean"], calib.loc[tag, "val_sd"]
    raw[idx], z[idx] = p, (p - mu) / sd
    span = hl["dt"].iloc[idx]
    print(f"{tag}: n={len(idx):6d}  {str(span.min())[:10]} -> {str(span.max())[:10]}"
          f"   raw sd {p.std():.5f}   z mean {z[idx].mean():+.3f} sd {z[idx].std():.3f}")
    del model; torch.cuda.empty_cache()

hl["predicted_sentiment"] = z
hl["predicted_sentiment_raw"] = raw
n_nan = int(hl["predicted_sentiment"].isna().sum())
print(f"\nscored {len(hl) - n_nan} of {len(hl)}; {n_nan} NaN (pre-2020, no model available)")

## Consistency check against the labelled-subset column

The 20 251 headlines that carry labels were scored in `oof_finbert_scoring.ipynb` by the same
checkpoints. Scoring them again here must reproduce those values, so the correlation on the overlap
should be 1.000 to numerical precision. Anything else means the fold assignment or the calibration
constants have drifted apart.

In [ ]:
prev = pd.read_csv(CLEANED / f"predicted_sentiments_finbert_oof_{RUN_TAG}_z.csv")
chk = prev.merge(hl[["timestamp", "title", "predicted_sentiment"]]
                   .rename(columns={"predicted_sentiment": "stream"}),
                 on=["timestamp", "title"], how="inner")
m = np.isfinite(chk["predicted_sentiment"]) & np.isfinite(chk["stream"])
r = float(np.corrcoef(chk.loc[m, "predicted_sentiment"], chk.loc[m, "stream"])[0, 1])
d = float(np.abs(chk.loc[m, "predicted_sentiment"] - chk.loc[m, "stream"]).max())
print(f"overlap {int(m.sum())} rows | corr {r:.6f} | max abs diff {d:.2e}")
assert r > 0.9999, "stream scores disagree with the labelled-subset column"
print("OK — identical scoring")

In [ ]:
hl[["timestamp", "title", "source", "predicted_sentiment"]].to_csv(FULL_CSV, index=False)
hl[["timestamp", "predicted_sentiment"]].to_csv(PS2_CSV, index=False)
hl[["timestamp", "title", "source", "predicted_sentiment", "predicted_sentiment_raw", "fold"]] \
    .to_csv(OUT / f"predicted_sentiments_oof_{RUN_TAG}_full_annotated.csv", index=False)

print(f"wrote {FULL_CSV}\n      {len(hl)} rows -> replaces predicted_sentiments_finbert.csv")
print(f"wrote {PS2_CSV}\n      2 columns -> replaces predicted_sentiments2.csv (backtesting/data_loader.py:27)")
print(f"wrote {OUT / f'predicted_sentiments_oof_{RUN_TAG}_full_annotated.csv'}")

old2 = pd.read_csv(CLEANED / "predicted_sentiments2.csv")
cmp = old2.merge(hl[["timestamp", "predicted_sentiment"]].rename(columns={"predicted_sentiment": "new"}),
                 on="timestamp", how="left")
m = np.isfinite(cmp["predicted_sentiment"]) & np.isfinite(cmp["new"])
print(f"\nvs published predicted_sentiments2.csv on {int(m.sum())} shared rows: "
      f"corr {float(np.corrcoef(cmp.loc[m, 'predicted_sentiment'], cmp.loc[m, 'new'])[0, 1]):+.4f}")
print(f"  published sd {cmp['predicted_sentiment'].std():.5f}   new sd {cmp['new'].std():.5f}")